# PIK Structural Credit Pricing

**Question:** What are the breakeven spreads between cash-pay, full-PIK, and PIK-toggle bonds across issuers of varying credit riskiness?

This notebook demonstrates the full structural credit + Monte Carlo pipeline:

1. **Merton Model** — Distance-to-default (DD), default probability (PD), and implied spreads from balance sheet data
2. **Endogenous Hazard** — Hazard rates that increase as PIK accrual raises leverage
3. **Dynamic Recovery** — Recovery rates that decline as notional grows
4. **Toggle Exercise** — Threshold-based and optimal (nested MC) PIK/cash decisions
5. **MC Pricing** — Full path simulation with first-passage default and PIK toggle logic

In [28]:
from __future__ import annotations
import math
from datetime import date, timedelta

from finstack import Money
from finstack.core.currency import USD
from finstack.core.market_data.context import MarketContext
from finstack.core.market_data.term_structures import DiscountCurve, HazardCurve
from finstack.valuations.instruments import (
    Bond,
    MertonModel,
    MertonAssetDynamics,
    MertonBarrierType,
    EndogenousHazardSpec,
    DynamicRecoverySpec,
    ToggleExerciseModel,
    MertonMcConfig,
    MertonMcResult,
)
from finstack.valuations.pricer import create_standard_registry

## 1. Setup: Bond and Market Parameters

In [29]:
# Market parameters
RISK_FREE = 0.045       # 4.5% base rate
COUPON_SPRD    = 0.04       # 400bp (typical HY PIK)
COUPON = RISK_FREE + COUPON_SPRD # 8.5% (typical HY PIK)
MATURITY  = 5           # years
NOTIONAL  = 100.0
AS_OF     = date(2025, 6, 15)
MAT_DATE  = AS_OF + timedelta(days=int(MATURITY * 365.25))
NUM_PATHS = 25_000
SEED      = 42

# Build cash and PIK bonds — PIK behavior comes from the bond's coupon_type
# or from pik_schedule on the MC config.
def _build_bond(coupon_type: str = "cash") -> Bond:
    return (
        Bond.builder(f"PIK-{coupon_type.upper()}")
        .money(Money(NOTIONAL, USD))
        .coupon_rate(COUPON)
        .coupon_type(coupon_type)
        .issue(AS_OF)
        .maturity(MAT_DATE)
        .frequency(2)
        .disc_id("USD-OIS")
        .credit_curve("CREDIT")
        .build()
    )

cash_bond = _build_bond("cash")
pik_bond = _build_bond("pik")
registry = create_standard_registry()

print(f"Bond: {MATURITY}Y  {COUPON:.1%} semi-annual  |  Notional: {NOTIONAL}  |  Maturity: {MAT_DATE}")

Bond: 5Y  8.5% semi-annual  |  Notional: 100.0  |  Maturity: 2030-06-15


## 2. Issuer Profiles

We define five issuers spanning the credit spectrum.  Each is characterised by:
- **Asset value (`V`)** — market value of the firm's assets (debt barrier = 100, so asset / 100 = coverage)
- **Asset vol (`vol`)** — annualised volatility of asset returns (higher vol = riskier)
- **Annual PD (`ann_pd`)** — historical annualised default probability used to calibrate the Merton barrier
- **Market spread (`market_spread`)** — observed cash-pay Z-spread; used to calibrate $\lambda_0$ via the reduced-form model
- **Base hazard (`h0`)** — $\lambda_0$ in the endogenous hazard model $\lambda(L) = \lambda_0 (L/L_0)^\beta$; calibrated from market spreads (or set directly)
- **Base recovery (`R0`)** — $R_0$ in the dynamic recovery model $R(N) = \max(\text{floor},\; R_0 \cdot N_0 / N)$; the expected recovery rate at par notional before any PIK accrual inflates it

### Calibrating h0 from market spreads

Given an observed cash-pay Z-spread $s$, we solve for the flat hazard rate $\lambda_0$ such that the survival-weighted bond price under $\lambda_0$ equals the price implied by $s$.  The first-order credit-triangle approximation is $\lambda_0 \approx s / (1 - R_0)$; the calibration below is exact for the bond's coupon structure and discrete semi-annual timing.

In [30]:
def calibrate_h0(target_spread: float, recovery: float) -> float:
    """Solve for the flat hazard rate whose reduced-form bond price
    matches the price implied by target_spread (Z-spread).

    Uses bisection over the survival-weighted PV formula:
      PV(h) = Σ cpn·D(t)·S(t) + N·D(T)·S(T) + R·N·Σ D(t)·[S(t-1)-S(t)]
    where D(t) = exp(-r·t) and S(t) = exp(-h·t).
    """
    n_periods = int(MATURITY * 2)
    times = [i / 2.0 for i in range(1, n_periods + 1)]
    cpn = COUPON / 2 * NOTIONAL

    target = sum(cpn * math.exp(-(RISK_FREE + target_spread) * t) for t in times)
    target += NOTIONAL * math.exp(-(RISK_FREE + target_spread) * MATURITY)

    def _hr_pv(h: float) -> float:
        pv = 0.0
        prev_s = 1.0
        for t in times:
            df = math.exp(-RISK_FREE * t)
            s = math.exp(-h * t)
            pv += cpn * df * s
            pv += recovery * NOTIONAL * df * (prev_s - s)
            prev_s = s
        pv += NOTIONAL * math.exp(-RISK_FREE * MATURITY) * math.exp(-h * MATURITY)
        return pv

    lo, hi = 0.0, 5.0
    for _ in range(200):
        mid = (lo + hi) / 2
        pv = _hr_pv(mid)
        if abs(pv - target) < 1e-6:
            return mid
        if pv > target:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2


# Quick sanity check: credit triangle approximation vs exact calibration
_test_s, _test_R = 0.0630, 0.30
_test_h = calibrate_h0(_test_s, _test_R)
print(f"Calibration check — spread={_test_s:.0%}, R={_test_R:.0%}")
print(f"  Exact:  h0 = {_test_h:.4%}")
print(f"  Approx: h0 ≈ s/(1-R) = {_test_s / (1 - _test_R):.4%}")

Calibration check — spread=6%, R=30%
  Exact:  h0 = 9.1076%
  Approx: h0 ≈ s/(1-R) = 9.0000%


In [31]:
# Set USE_MARKET_SPREADS = True  to calibrate h0 from observed spreads.
# Set USE_MARKET_SPREADS = False to use hand-picked h0 values directly.
USE_MARKET_SPREADS = True

PROFILES = [
    {"name": "BB+ (Solid HY)",       "V": 200, "vol": 0.20, "ann_pd": 0.0020, "market_spread": 0.0085,  "R0": 0.45},
    {"name": "BB- (Mid HY)",         "V": 165, "vol": 0.25, "ann_pd": 0.0100, "market_spread": 0.0210,  "R0": 0.40},
    {"name": "B  (Weak HY)",         "V": 140, "vol": 0.30, "ann_pd": 0.0250, "market_spread": 0.0390,  "R0": 0.35},
    {"name": "B- (Stressed)",        "V": 125, "vol": 0.35, "ann_pd": 0.0550, "market_spread": 0.0630,  "R0": 0.30},
    {"name": "CCC (Deeply Stressed)","V": 115, "vol": 0.40, "ann_pd": 0.1000, "market_spread": 0.1050,  "R0": 0.25},
]

if USE_MARKET_SPREADS:
    for p in PROFILES:
        p["h0"] = calibrate_h0(p["market_spread"], p["R0"])
else:
    _DIRECT_H0 = [0.015, 0.035, 0.060, 0.090, 0.140]
    for p, h in zip(PROFILES, _DIRECT_H0):
        p["h0"] = h

print(f"{'Issuer':<25s}  {'Assets':>6s}  {'Vol':>5s}  {'Mkt Sprd':>8s}  {'h0 (cal)':>8s}  {'h≈s/(1-R)':>9s}  {'R0':>5s}")
print("-" * 80)
for p in PROFILES:
    s = p["market_spread"]
    approx = s / (1 - p["R0"])
    print(f"{p['name']:<25s}  {p['V']:>6.0f}  {p['vol']:>5.0%}  {s * 1e4:>7.0f}bp  "
          f"{p['h0']:>8.2%}  {approx:>9.2%}  {p['R0']:>5.0%}")

Issuer                     Assets    Vol  Mkt Sprd  h0 (cal)  h≈s/(1-R)     R0
--------------------------------------------------------------------------------
BB+ (Solid HY)                200    20%       85bp     1.43%      1.55%    45%
BB- (Mid HY)                  165    25%      210bp     3.34%      3.50%    40%
B  (Weak HY)                  140    30%      390bp     5.91%      6.00%    35%
B- (Stressed)                 125    35%      630bp     9.11%      9.00%    30%
CCC (Deeply Stressed)         115    40%     1050bp    14.68%     14.00%    25%


## 3. Merton Model: Analytical Credit Metrics

Before running Monte Carlo, the Merton model gives us closed-form distance-to-default and default probability.  The model treats equity as a call option on firm assets with strike = debt barrier.

In [32]:
print(f"{'Issuer':<25s}  {'Barrier':>8s}  {'DD':>6s}  {'PD(5Y)':>7s}  {'Impl Sprd':>9s}")
print("-" * 65)

for p in PROFILES:
    five_year_pd = 1.0 - math.exp(-p["ann_pd"] * MATURITY)
    m = MertonModel.from_target_pd(
        asset_value=p["V"],
        asset_vol=p["vol"],
        risk_free_rate=RISK_FREE,
        target_pd=five_year_pd,
        maturity=MATURITY,
    )
    dd = m.distance_to_default(MATURITY)
    pd = m.default_probability(MATURITY)
    spread = m.implied_spread(MATURITY, p["R0"])

    print(f"{p['name']:<25s}  {m.debt_barrier:>8.1f}  {dd:>6.2f}  {pd:>7.2%}  {spread * 10_000:>7.0f} bp")

Issuer                      Barrier      DD   PD(5Y)  Impl Sprd
-----------------------------------------------------------------
BB+ (Solid HY)                 80.0    2.33    1.00%       11 bp
BB- (Mid HY)                   70.0    1.66    4.88%       59 bp
B  (Weak HY)                   63.1    1.19   11.75%      159 bp
B- (Stressed)                  66.4    0.70   24.04%      369 bp
CCC (Deeply Stressed)          75.8    0.27   39.35%      699 bp


## 4. Endogenous Hazard and Dynamic Recovery

PIK bonds create a **feedback loop**: PIK accrual grows the notional, which increases leverage, which raises the hazard rate and lowers recovery.  These two modules capture this:

- **Endogenous Hazard** (power law): $\lambda(L) = \lambda_0 \cdot (L / L_0)^\beta$
- **Dynamic Recovery** (floored inverse): $R(N) = \max(\text{floor}, R_0 \cdot N_0 / N)$

In [33]:
# Demonstrate the hazard feedback for the B- issuer
p = PROFILES[3]  # B- (Stressed)
five_year_pd = 1.0 - math.exp(-p["ann_pd"] * MATURITY)
m = MertonModel.from_target_pd(p["V"], p["vol"], RISK_FREE, five_year_pd, MATURITY)
base_lev = m.debt_barrier / p["V"]
endo = EndogenousHazardSpec.power_law(
    base_hazard=p["h0"],
    base_leverage=base_lev,
    exponent=2.0,
)

dyn_rec = DynamicRecoverySpec.floored_inverse(
    base_recovery=p["R0"],
    base_notional=NOTIONAL,
    floor=0.10,
)

print(f"B- Issuer:  barrier = {m.debt_barrier:.1f},  base leverage = {base_lev:.2f},  base hazard = {p['h0']:.1%}")
print()
print(f"{'Leverage':>10s}  {'Hazard':>8s}  |  {'Notional':>10s}  {'Recovery':>8s}")
print("-" * 48)
for lev in [0.60, 0.70, 0.80, 0.90, 1.00, 1.10, 1.20]:
    h = endo.hazard_at_leverage(lev)
    ntl = NOTIONAL * (lev / base_lev)  # implied notional at this leverage
    r = dyn_rec.recovery_at_notional(ntl)
    print(f"{lev:>10.2f}  {h:>8.2%}  |  {ntl:>10.1f}  {r:>8.2%}")

B- Issuer:  barrier = 66.4,  base leverage = 0.53,  base hazard = 9.1%

  Leverage    Hazard  |    Notional  Recovery
------------------------------------------------
      0.60    11.63%  |       113.0    26.55%
      0.70    15.83%  |       131.8    22.76%
      0.80    20.67%  |       150.7    19.91%
      0.90    26.16%  |       169.5    17.70%
      1.00    32.30%  |       188.3    15.93%
      1.10    39.08%  |       207.1    14.48%
      1.20    46.51%  |       226.0    13.28%


## 5. Monte Carlo Pricing: Cash vs Full PIK vs Toggle

We now run the full MC engine for each issuer and each bond structure:

1. **Cash-Pay** — No PIK features, plain structural credit MC
2. **Full PIK** — All coupons accrete to notional; endogenous hazard + dynamic recovery enabled
3. **PIK Toggle** — Borrower elects PIK when hazard rate exceeds 10% threshold

In [34]:
def make_merton(p: dict) -> MertonModel:
    """Calibrate Merton barrier from historical annual PD."""
    five_year_pd = 1.0 - math.exp(-p["ann_pd"] * MATURITY)
    return MertonModel.from_target_pd(
        asset_value=p["V"],
        asset_vol=p["vol"],
        risk_free_rate=RISK_FREE,
        target_pd=five_year_pd,
        maturity=MATURITY,
    )


def make_config(
    p: dict,
    pik_schedule: str | list | None = None,
    toggle: ToggleExerciseModel | None = None,
) -> MertonMcConfig:
    """Build an MC config with endogenous hazard + dynamic recovery."""
    m = make_merton(p)
    endo = EndogenousHazardSpec.power_law(p["h0"], m.debt_barrier / p["V"], 2.0)
    drec = DynamicRecoverySpec.floored_inverse(p["R0"], NOTIONAL, 0.10)

    kw: dict = dict(
        merton=m,
        pik_schedule=pik_schedule,
        endogenous_hazard=endo,
        dynamic_recovery=drec,
        num_paths=NUM_PATHS,
        seed=SEED,
        antithetic=True,
        time_steps_per_year=12,
    )
    if toggle is not None:
        kw["toggle_model"] = toggle
    return MertonMcConfig(**kw)


def make_bond(coupon_type: str, mc_config: MertonMcConfig) -> Bond:
    """Build a bond with MC config attached for registry pricing."""
    return (
        Bond.builder(f"PIK-{coupon_type.upper()}")
        .money(Money(NOTIONAL, USD))
        .coupon_rate(COUPON)
        .coupon_type(coupon_type)
        .issue(AS_OF)
        .maturity(MAT_DATE)
        .frequency(2)
        .disc_id("USD-OIS")
        .credit_curve("CREDIT")
        .merton_mc(mc_config)
        .build()
    )


market = build_market() if 'build_market' in dir() else MarketContext()
# Build flat discount curve in the market
if not hasattr(market, '_has_disc'):
    market = MarketContext()
market.insert_discount(DiscountCurve(
    "USD-OIS", AS_OF,
    [(t, math.exp(-RISK_FREE * t))
     for t in [0.0, 0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0]],
))



# No compute_z_spread needed — the MC pricer computes cash-equivalent
# Z-spread and YTM internally and returns them in result.measures.

In [35]:
# Run the MC engine via the pricer registry for all profiles
toggle = ToggleExerciseModel.threshold("hazard_rate", 0.10, "above")
results = {}  # {issuer_name: {structure: dict}}

for p in PROFILES:
    base_cfg   = make_config(p)
    toggle_cfg = make_config(p, pik_schedule="toggle", toggle=toggle)

    cash_bond   = make_bond("cash", base_cfg)
    pik_bond    = make_bond("pik", base_cfg)
    toggle_bond = make_bond("cash", toggle_cfg)

    def _mc(bond: Bond) -> dict:
        r = registry.price(bond, "merton_mc", market, as_of=AS_OF)
        pv = r.value.amount
        m = r.measures
        return {"price_pct": pv / NOTIONAL * 100,
                "z_spread_bp": m.get("z_spread", 0) * 10_000,
                "ytm_pct": m.get("ytm", 0) * 100,
                "expected_loss": m.get("expected_loss", 0),
                "default_rate": m.get("default_rate", 0), "pik_fraction": m.get("pik_fraction", 0),
                "avg_terminal_notional": m.get("avg_terminal_notional", NOTIONAL),
                "mc_standard_error": m.get("mc_standard_error", 0)}

    results[p["name"]] = {
        "Cash-Pay":   _mc(cash_bond),
        "Full PIK":   _mc(pik_bond),
        "PIK Toggle": _mc(toggle_bond),
    }

print(f"Priced {len(PROFILES)} issuers x 3 structures = {len(PROFILES) * 3} MC runs")

Priced 5 issuers x 3 structures = 15 MC runs


## 6. Results: Detailed View

In [36]:
for issuer, structs in results.items():
    print(f"\n{'=' * 90}")
    print(f"  {issuer}")
    print(f"{'=' * 90}")
    print(f"  {'Structure':<15s}  {'Price':>7s}  {'Z-Sprd':>7s}  {'E[Loss]':>7s}  "
          f"{'DefRate':>7s}  {'PIK%':>6s}  {'TermNtl':>8s}  {'SE':>6s}")
    print(f"  {'-' * 75}")
    for name, r in structs.items():
        print(f"  {name:<15s}  {r['price_pct']:7.2f}  {r['z_spread_bp']:6.0f}bp  "
              f"{r['expected_loss']:7.2%}  {r['default_rate']:7.2%}  {r['pik_fraction']:5.1%}  "
              f"{r['avg_terminal_notional']:8.1f}  {r['mc_standard_error']:6.4f}")


  BB+ (Solid HY)
  Structure          Price   Z-Sprd  E[Loss]  DefRate    PIK%   TermNtl      SE
  ---------------------------------------------------------------------------
  Cash-Pay          116.48      20bp    0.86%    1.94%   0.0%     100.0  0.0005
  Full PIK          119.46     -39bp   -1.68%    1.94%  100.0%     151.6  0.0007
  PIK Toggle        116.41      22bp    0.92%    1.94%   0.6%     100.2  0.0005

  BB- (Mid HY)
  Structure          Price   Z-Sprd  E[Loss]  DefRate    PIK%   TermNtl      SE
  ---------------------------------------------------------------------------
  Cash-Pay          112.12     110bp    4.56%    9.12%   0.0%     100.0  0.0011
  Full PIK          113.17      88bp    3.68%    9.12%  100.0%     151.6  0.0016
  PIK Toggle        111.20     130bp    5.35%    9.12%  28.6%     112.4  0.0014

  B  (Weak HY)
  Structure          Price   Z-Sprd  E[Loss]  DefRate    PIK%   TermNtl      SE
  ----------------------------------------------------------------------

## 7. Breakeven Spread Summary

The key question: **How much extra spread should an investor demand for PIK risk?**

In [37]:
print(f"{'Issuer':<25s}  {'Cash':>8s}  {'PIK':>8s}  {'Toggle':>8s}  "
      f"{'PIK-Cash':>8s}  {'Tog-Cash':>8s}")
print("-" * 80)

for issuer, structs in results.items():
    c = structs["Cash-Pay"]["z_spread_bp"]
    p = structs["Full PIK"]["z_spread_bp"]
    t = structs["PIK Toggle"]["z_spread_bp"]
    print(f"{issuer:<25s}  {c:>7.0f}bp {p:>7.0f}bp {t:>7.0f}bp  {p - c:>+8.0f}  {t - c:>+8.0f}")

print()
print("All values are CASH-EQUIVALENT Z-spreads: the Z-spread on a standard")
print("cash-pay bond that reproduces each structure's MC price.")
print("PIK-Cash  = PIK premium in Z-spread terms (positive = PIK is riskier)")
print("Tog-Cash  = toggle premium in Z-spread terms")

Issuer                         Cash       PIK    Toggle  PIK-Cash  Tog-Cash
--------------------------------------------------------------------------------
BB+ (Solid HY)                  20bp     -39bp      22bp       -59        +1
BB- (Mid HY)                   110bp      88bp     130bp       -22       +20
B  (Weak HY)                   292bp     329bp     346bp       +37       +54
B- (Stressed)                  710bp     851bp     862bp      +141      +151
CCC (Deeply Stressed)         1497bp    1759bp    1763bp      +262      +266

All values are CASH-EQUIVALENT Z-spreads: the Z-spread on a standard
cash-pay bond that reproduces each structure's MC price.
PIK-Cash  = PIK premium in Z-spread terms (positive = PIK is riskier)
Tog-Cash  = toggle premium in Z-spread terms


## 7b. Hazard-Rate Model: PIK Pricing Without Structural Feedback

Before the full Merton simulation, we can ask a simpler question: **how much of the PIK premium comes purely from the timing and notional effects?**

A **flat hazard rate** model uses constant $\lambda$ and the library's `HazardCurve` and `DiscountCurve` term structures:

- `HazardCurve.survival(t)` $= S(t) = e^{-\lambda t}$ — probability of no default by time $t$
- `HazardCurve.default_prob(t_1, t_2)` $= S(t_1) - S(t_2)$ — default probability in each period
- `DiscountCurve.df(t)` $= D(t) = e^{-r t}$ — risk-free discount factor

**Cash-pay:** Coupons arrive at $t_1, \ldots, t_n$ — each weighted by $S(t_i) \cdot D(t_i)$, plus principal at maturity.
**Full PIK:** No coupon cash flows. At maturity, receive $N \cdot (1 + c/f)^n$ — the entire PIK-inflated notional, weighted by $S(T) \cdot D(T)$.

The PIK bond concentrates all value at the longest maturity (lowest survival probability) with a larger notional (higher loss given default). This captures the **timing + notional** effect but **not** the endogenous feedback loop (PIK → higher leverage → higher $\lambda$).

We show two views:
1. **Price comparison** — at each issuer's base $\lambda$, how do HR prices compare to MC prices?
2. **Implied hazard rates** — given the MC model's prices, what flat $\lambda$ reproduces them? The PIK implied $\lambda$ exceeds the cash implied $\lambda$ by the **structural hazard premium**.

In [ ]:
import polars as pl

# Show the PIK cashflow schedule for a representative issuer (B- Stressed)
# to illustrate the difference in cashflow timing and notional between
# cash-pay and PIK bonds that the hazard-rate model must capture.

p = PROFILES[3]  # B- (Stressed)
hr_market = MarketContext()
hr_market.insert_discount(DiscountCurve(
    "USD-OIS", AS_OF,
    [(t, math.exp(-RISK_FREE * t))
     for t in [0.0, 0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0]],
))
hr_market.insert_hazard(HazardCurve(
    "CREDIT", AS_OF, [(0.0, p["h0"]), (10.0, p["h0"])],
    recovery_rate=p["R0"],
))

print(f"B- Issuer: flat hazard = {p['h0']:.2%},  recovery = {p['R0']:.0%}")
print()

for label, bond in [("CASH-PAY", cash_bond), ("FULL PIK", pik_bond)]:
    df = bond.pricing_cashflows(hr_market, as_of=AS_OF)
    display_df = df.select([
        pl.col("pay_date"),
        pl.col("kind"),
        pl.col("amount").round(2),
        pl.col("discount_factor").round(6),
        pl.col("survival_prob").round(6),
        pl.col("pv").round(2),
    ])
    print(f"── {label} Cashflow Schedule ──")
    print(display_df)
    print(f"  Total PV = {df['pv'].sum():.2f}")
    print()

In [38]:
# -- Hazard-rate pricing using library HazardBondEngine via registry ------
#
# The library's HazardBondEngine computes survival-weighted PV for the alive
# leg plus a FRP (fractional recovery of par) default leg.  For the PIK bond,
# the cashflow builder generates the accreted notional schedule automatically.

def hr_price(bond: Bond, hazard: float, recovery: float) -> float:
    """Price a bond using the library's hazard-rate engine."""
    market = MarketContext()
    market.insert_discount(DiscountCurve(
        "USD-OIS", AS_OF,
        [(t, math.exp(-RISK_FREE * t))
         for t in [0.0, 0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0]],
    ))
    market.insert_hazard(HazardCurve(
        "CREDIT", AS_OF, [(0.0, hazard), (10.0, hazard)],
        recovery_rate=recovery,
    ))
    return registry.price(bond, "hazard_rate", market, as_of=AS_OF).value.amount

def find_implied_hazard(bond: Bond, target: float, recovery: float) -> float:
    """Bisection: find flat lambda that prices to target PV."""
    lo, hi = 0.0, 5.0
    for _ in range(200):
        mid = (lo + hi) / 2
        pv = hr_price(bond, mid, recovery)
        if abs(pv - target) < 1e-6:
            return mid
        if pv > target:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2

# -- View 1: prices at each issuer's base hazard rate --------------------

print("HAZARD-RATE PRICES: Cash-Pay vs PIK at Each Issuer's Flat lambda")
print("=" * 100)
print(f"  {'Issuer':<25s}  {'lam(bp)':>7s}  "
      f"{'HR Cash':>9s}  {'HR PIK':>9s}  {'d Price':>8s}  "
      f"{'MC Cash':>9s}  {'MC PIK':>9s}  {'d Price':>8s}")
print("  " + "-" * 90)

for p in PROFILES:
    lam, rec = p["h0"], p["R0"]
    hr_cash = hr_price(cash_bond, lam, rec)
    hr_pik  = hr_price(pik_bond, lam, rec)

    mc_c = results[p["name"]]["Cash-Pay"]["price_pct"]
    mc_p = results[p["name"]]["Full PIK"]["price_pct"]

    print(f"  {p['name']:<25s}  {lam*1e4:7.0f}  "
          f"{hr_cash:9.2f}  {hr_pik:9.2f}  {hr_pik - hr_cash:+8.2f}  "
          f"{mc_c:9.2f}  {mc_p:9.2f}  {mc_p - mc_c:+8.2f}")

print()
print("  d Price = PIK minus Cash (negative = PIK trades cheaper)")
print("  HR captures timing + notional; MC adds endogenous feedback.")

# -- View 2: implied hazard rates from MC prices -------------------------

print()
print("IMPLIED HAZARD RATES: Flat lambda Backing Out Each Merton MC Price")
print("=" * 70)
print(f"  {'Issuer':<25s}  {'Base lam':>8s}  "
      f"{'lam Cash':>8s}  {'lam PIK':>8s}  {'d lam':>7s}")
print("  " + "-" * 60)

# Build plain bonds without MC config for hazard-rate implied hazard solving
cash_bond_hr = _build_bond("cash")
pik_bond_hr = _build_bond("pik")

for p in PROFILES:
    rec = p["R0"]
    mc_cash = results[p["name"]]["Cash-Pay"]
    mc_pik  = results[p["name"]]["Full PIK"]

    cash_target = mc_cash["price_pct"] / 100 * NOTIONAL
    pik_target  = mc_pik["price_pct"] / 100 * NOTIONAL

    lam_c = find_implied_hazard(cash_bond_hr, cash_target, rec)
    lam_p = find_implied_hazard(pik_bond_hr, pik_target, rec)

    print(f"  {p['name']:<25s}  {p['h0']*1e4:8.0f}  "
          f"{lam_c*1e4:8.0f}  {lam_p*1e4:8.0f}  {(lam_p - lam_c)*1e4:+7.0f}")

print()
print("  lam Cash / lam PIK = flat hazard rates reproducing the MC model prices")
print("  d lam = PIK hazard premium (extra hazard the structural model")
print("          implicitly assigns to PIK over cash-pay)")

HAZARD-RATE PRICES: Cash-Pay vs PIK at Each Issuer's Flat lambda
  Issuer                     lam(bp)    HR Cash     HR PIK   d Price    MC Cash     MC PIK   d Price
  ------------------------------------------------------------------------------------------
  BB+ (Solid HY)                 143     113.35     110.89     -2.46     116.48     119.46     +2.98
  BB- (Mid HY)                   334     107.56     103.76     -3.80     112.12     113.17     +1.05
  B  (Weak HY)                   591      99.76      94.44     -5.32     103.87     102.26     -1.61
  B- (Stressed)                  911      90.33      83.45     -6.88      87.28      82.39     -4.89
  CCC (Deeply Stressed)         1468      76.13      67.34     -8.79      63.60      57.43     -6.17

  d Price = PIK minus Cash (negative = PIK trades cheaper)
  HR captures timing + notional; MC adds endogenous feedback.

IMPLIED HAZARD RATES: Flat lambda Backing Out Each Merton MC Price
  Issuer                     Base lam  lam Cas

## 8. Sensitivity: PIK Premium vs Asset Coverage

Sweep asset values from 110 to 220 (coverage 1.10x to 2.20x) and plot the PIK premium.

In [39]:
# Sweep coverage ratios
coverages = []
cash_prices = []
pik_prices = []
pik_discounts = []

for V in range(110, 225, 5):
    p = {"V": V, "vol": 0.30, "ann_pd": 0.02, "h0": 0.06, "R0": 0.35}
    cfg = make_config(p)

    cb = make_bond("cash", cfg)
    pb = make_bond("pik", cfg)
    rc = registry.price(cb, "merton_mc", market, as_of=AS_OF).value.amount / NOTIONAL * 100
    rp = registry.price(pb, "merton_mc", market, as_of=AS_OF).value.amount / NOTIONAL * 100

    coverages.append(V / 100)
    cash_prices.append(rc)
    pik_prices.append(rp)
    pik_discounts.append(rc - rp)

print(f"{'Coverage':>10s}  {'Cash':>8s}  {'PIK':>8s}  {'Discount':>8s}")
print("-" * 40)
for i in range(0, len(coverages), 3):
    print(f"{coverages[i]:>10.2f}x {cash_prices[i]:>8.2f}  {pik_prices[i]:>8.2f}  {pik_discounts[i]:>+8.2f}")

  Coverage      Cash       PIK  Discount
----------------------------------------
      1.10x   106.70    105.95     +0.75
      1.25x   106.70    105.95     +0.75
      1.40x   106.70    105.95     +0.75
      1.55x   106.70    105.95     +0.75
      1.70x   106.70    105.95     +0.75
      1.85x   106.70    105.95     +0.75
      2.00x   106.70    105.95     +0.75
      2.15x   106.70    105.95     +0.75


## 9. Toggle Strategy Comparison

Compare three toggle strategies for the B- issuer:
- **Threshold (hazard > 10%)** — Simple rule-based toggle
- **Stochastic (sigmoid)** — Smooth probability function of hazard rate
- **Optimal Exercise (nested MC)** — Forward-looking equity-maximising decision

In [40]:
p = PROFILES[3]  # B- (Stressed)

toggle_strategies = {
    "Threshold (h > 10%)":  ToggleExerciseModel.threshold("hazard_rate", 0.10, "above"),
    "Stochastic (sigmoid)": ToggleExerciseModel.stochastic("hazard_rate", -3.0, 25.0),
    "Optimal (nested MC)":  ToggleExerciseModel.optimal_exercise(
        nested_paths=200, equity_discount_rate=COUPON / 2, asset_vol=p["vol"], risk_free_rate=RISK_FREE,
    ),
}

m = make_merton(p)
print(f"B- Issuer: V={p['V']}, vol={p['vol']:.0%}, barrier={m.debt_barrier:.1f}")
print()
print(f"  {'Strategy':<25s}  {'Price':>7s}  {'E[Loss]':>7s}  "
      f"{'DefRate':>7s}  {'PIK%':>6s}  {'TermNtl':>8s}")
print(f"  {'-' * 60}")

def _show(label, bond):
    r = registry.price(bond, "merton_mc", market, as_of=AS_OF)
    pv = r.value.amount / NOTIONAL * 100
    ms = r.measures
    print(f"  {label:<25s}  {pv:7.2f}  "
          f"{ms.get('expected_loss', 0):7.2%}  {ms.get('default_rate', 0):7.2%}  "
          f"{ms.get('pik_fraction', 0):5.1%}  {ms.get('avg_terminal_notional', NOTIONAL):8.1f}")

base_cfg = make_config(p)
_show("Cash-Pay", make_bond("cash", base_cfg))
_show("Full PIK", make_bond("pik", base_cfg))

for label, tog in toggle_strategies.items():
    tog_cfg = make_config(p, pik_schedule="toggle", toggle=tog)
    _show(label, make_bond("cash", tog_cfg))

B- Issuer: V=125, vol=35%, barrier=66.4

  Strategy                     Price  E[Loss]  DefRate    PIK%   TermNtl
  ------------------------------------------------------------
  Cash-Pay                     87.28   25.71%   41.08%   0.0%     100.0
  Full PIK                     82.39   29.87%   41.08%  100.0%     151.6
  Threshold (h > 10%)          82.04   30.17%   41.08%  79.7%     137.3
  Stochastic (sigmoid)         82.23   30.00%   41.08%  72.6%     132.9
  Optimal (nested MC)          83.40   29.01%   41.08%  39.5%     114.8


## 10. Sensitivity: Asset Volatility Impact

For a fixed leverage (1.40x coverage), sweep asset vol from 15% to 45% to see how vol amplifies PIK risk.

In [41]:
print(f"{'Vol':>5s}  {'Cash Price':>10s}  {'PIK Price':>10s}  {'Discount':>10s}")
print("-" * 45)

for vol_pct in range(15, 50, 5):
    vol = vol_pct / 100
    p = {"V": 140, "vol": vol, "ann_pd": 0.02, "h0": 0.06, "R0": 0.35}
    cfg = make_config(p)

    cb = make_bond("cash", cfg)
    pb = make_bond("pik", cfg)
    rc = registry.price(cb, "merton_mc", market, as_of=AS_OF).value.amount / NOTIONAL * 100
    rp = registry.price(pb, "merton_mc", market, as_of=AS_OF).value.amount / NOTIONAL * 100

    print(f"{vol:>5.0%}  {rc:>10.2f}  {rp:>10.2f}  {rc - rp:>+10.2f}")

  Vol  Cash Price   PIK Price    Discount
---------------------------------------------
  15%      101.71      100.71       +1.00
  20%      104.58      103.69       +0.89
  25%      105.89      105.07       +0.82
  30%      106.70      105.95       +0.75
  35%      107.24      106.51       +0.73
  40%      107.65      106.94       +0.71
  45%      108.00      107.33       +0.67


## Conclusions

**Key findings:**

1. **PIK premium is non-linear in credit quality.** For healthy issuers (BB+, 2x coverage), PIK barely matters — defaults are rare, so PIK accrual doesn’t spiral.  But for stressed issuers (CCC, 1.15x coverage), the PIK premium can exceed 500bp.

2. **The feedback loop drives PIK risk.** The combination of endogenous hazard (leverage raises default intensity) and dynamic recovery (higher notional dilutes recovery) creates a self-reinforcing spiral that dramatically worsens PIK bond economics for weak credits.

3. **Simple hazard rates miss the story.** A flat hazard-rate model captures the timing and notional effects of PIK but grossly underestimates the premium for stressed issuers.  For CCC credits, the implied hazard rate for PIK is ~700bp above the cash implied hazard — almost entirely driven by the structural feedback that simple models omit.

4. **Toggle is not free.** Even though the toggle preserves the borrower’s *option* to pay cash, the investor still bears negative convexity: the borrower PIKs precisely when credit deteriorates.  The toggle spread premium sits between cash and full PIK.

5. **Volatility amplifies PIK risk.** Higher asset vol increases both the frequency and severity of the leverage spiral, making PIK discounts disproportionately larger.

6. **Barriers are calibrated to realistic PDs.** Using `MertonModel.from_target_pd` with historical annualised PDs (BB ~20bp, B ~200bp, CCC ~400bp) produces calibrated barriers that give realistic 5-year default probabilities and spreads.

7. **Toggle can be worse than full PIK for investors (adverse selection).** Toggle concentrates PIK accrual on the worst paths (high hazard triggers the toggle), while full PIK distributes it uniformly. The endogenous leverage spiral is more intense when seeded on already-stressed paths, so toggle can produce a larger investor loss than mandatory full PIK.